Random Forest analysis

In [10]:
import pandas as pd

data = pd.read_csv("./Training_Data.csv")
data.describe()

,number_of_maps,firepower,entrying,trading,opening,clutching,sniping,utility,igl
count,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000
mean,50.108571,54.708571,48.554286,47.342857,53.548571,50.885714,20.542857,58.771429,0.194286
std,16.175925,24.174668,25.217586,20.248943,22.222146,17.346985,37.558493,18.594409,0.396785
min,14.000000,2.000000,5.000000,6.000000,5.000000,18.000000,0.000000,17.000000,0.000000
25%,37.000000,35.000000,29.500000,31.500000,35.000000,38.000000,0.000000,45.500000,0.000000
50%,51.000000,55.000000,47.000000,47.000000,52.000000,48.000000,0.000000,58.000000,0.000000
75%,58.000000,73.500000,69.000000,62.000000,75.000000,65.000000,3.500000,72.500000,0.000000
max,106.000000,100.000000,98.000000,96.000000,97.000000,91.000000,95.000000,95.000000,1.000000


In [11]:
feature_cols = [
    "firepower",
    "entrying",
    "trading",
    "opening",
    "clutching",
    "sniping",
    "utility",
    "igl",
]

x = data[feature_cols]
y = data["position"]

# can comment line below out
y.replace("Linchpin", "Closer", inplace=True)

y.head()
y.describe()

count        175
unique         3
top       Closer
freq          72
Name: position, dtype: object

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

In [13]:
rf = RandomForestClassifier(n_estimators=5000, max_depth=5)
rf.fit(x_train, y_train)

,n_estimators,5000
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
y_pred = rf.predict(x_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8409090909090909


In [16]:
param_dist = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 8),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5),
}


# Create a random forest classifier
rf = RandomForestClassifier(random_state=42, n_jobs=1)

# Use random search to find the best hyperparameters
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
).fit(x_train, y_train)

# Create a variable for the best model
best_rf = rand_search.best_estimator_

params = rand_search.best_params_

# Print the best hyperparameters
print("Best hyperparameters:", params)

Best hyperparameters: {'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 261}


In [18]:
opt_rf = RandomForestClassifier(**params)
opt_rf.fit(x_train, y_train)

,n_estimators,261
,criterion,'gini'
,max_depth,4
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
y_pred = opt_rf.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8636363636363636
